In [0]:
source_path='/Volumes/finguard/source/fraud_watchlist/source_data/'

In [0]:
dbutils.fs.ls(source_path)

In [0]:
input_stream=(spark.readStream
              .format('cloudFiles')
              .option('cloudFiles.format','json')
              .option('cloudFiles.schemaLocation','/Volumes/finguard/source/fraud_watchlist/schema/')
              .option('cloudFiles.inferColumnTypes','true')
              .load(source_path)
)

In [0]:
from pyspark.sql import functions as F
transformed_df=input_stream.select(
    "*",
    F.col("_metadata.file_path").alias("file_path"),
    F.current_timestamp().alias("ingestion_timestamp")
)

In [0]:
streaming_query=(transformed_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointlocation", "/Volumes/finguard/source/fraud_watchlist/checkpoint/")
.trigger(availableNow=True)
.toTable("finguard.bronze.fraud_watchlist_batch_test")
)


In [0]:
%sql
select * from finguard.bronze.fraud_watchlist_batch_test